### 1. Setup & Library Imports

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import plotly.express as px
plt.rcParams['figure.dpi'] = 150

### 2. Load Dataset & Exploratory Data Analysis 

#### 2.1 Loading original datasets

In [10]:
df_train = pd.read_csv('fraudTrain.csv')
df_test = pd.read_csv('fraudTest.csv')

#### 2.2 Adding datasets flags

In [11]:
df_train['data_type'] = 'train'
df_test['data_type'] = 'test'

#### 2.3 Creating the masters dataset

In [12]:
transactions_master = pd.concat([df_train, df_test], ignore_index=True)
print(f"✅ Master dataset: {transactions_master.shape}")
print(transactions_master.head(3))

✅ Master dataset: (1604294, 24)
   ID trans_date_trans_time        cc_num                         merchant  \
0   0         1/1/2019 0:00  2.703190e+15       fraud_Rippin, Kub and Mann   
1   1         1/1/2019 0:00  6.304230e+11  fraud_Heller, Gutmann and Zieme   
2   2         1/1/2019 0:00  3.885950e+13             fraud_Lind-Buckridge   

        category     amt      first     last gender  \
0       misc_net    4.97   Jennifer    Banks      F   
1    grocery_pos  107.23  Stephanie     Gill      F   
2  entertainment  220.11     Edward  Sanchez      M   

                         street  ...      long city_pop  \
0                561 Perry Cove  ...  -81.1781     3495   
1  43039 Riley Greens Suite 393  ... -118.2105      149   
2      594 White Dale Suite 530  ... -112.2620     4154   

                                 job        dob  \
0          Psychologist, counselling   3/9/1988   
1  Special educational needs teacher  6/21/1978   
2        Nature conservation officer  1/19/1

#### 2.4 Fraud Rate Check

In [13]:
print("\nFraud distribution:")
print(transactions_master.is_fraud.value_counts(normalize=True))
print(f"Train fraud: {df_train.is_fraud.mean():.3%}")
print(f"Test fraud:  {df_test.is_fraud.mean():.3%}")


Fraud distribution:
is_fraud
0    0.994919
1    0.005081
Name: proportion, dtype: float64
Train fraud: 0.573%
Test fraud:  0.386%


#### 2.5 Dataset Imbalance + Dimensions

In [18]:
# Cell 8: Dataset Imbalance + Dimensions  
print("📊 DATASET DIMENSIONS & IMBALANCE")
print("="*50)

for split in ['train', 'test', 'All']:
    subset = transactions_master[transactions_master.data_type == split] if split != 'All' else transactions_master
    
    fraud_rate = subset.is_fraud.mean()
    imbalance_ratio = 1 / fraud_rate  # How many legit txns per fraud
    
    print(f"{split:>6}: {subset.shape[0]:>8,} txns | "
          f"Fraud: {fraud_rate:.3%} | "
          f"Ratio: 1:{imbalance_ratio:.0f}")

print(f"\nCards: {transactions_master.cc_num.nunique():,} | "
      f"Merchants: {transactions_master.merchant.nunique():,}")
print("train/test split:", transactions_master.data_type.value_counts().to_dict())

📊 DATASET DIMENSIONS & IMBALANCE
 train: 1,048,575 txns | Fraud: 0.573% | Ratio: 1:175
  test:  555,719 txns | Fraud: 0.386% | Ratio: 1:259
   All: 1,604,294 txns | Fraud: 0.508% | Ratio: 1:197

Cards: 959 | Merchants: 693
train/test split: {'train': 1048575, 'test': 555719}


#### 2.6 Summary Statistics

In [15]:
print("\nTransaction amounts:")
print(transactions_master.amt.describe())


Transaction amounts:
count    1.604294e+06
mean     6.997209e+01
std      1.588492e+02
min      1.000000e+00
25%      9.640000e+00
50%      4.739000e+01
75%      8.304000e+01
max      2.894890e+04
Name: amt, dtype: float64


#### 2.7 Data Quality

In [16]:
print("\nMissing values:")
print(transactions_master.isnull().sum())


Missing values:
ID                       0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
data_type                0
dtype: int64


### 3. Feature Engineering

#### 3.1 Time features

In [19]:
transactions_master['trans_datetime'] = pd.to_datetime(transactions_master.trans_date_trans_time)
transactions_master['hour'] = transactions_master.trans_datetime.dt.hour
transactions_master['day_of_week'] = transactions_master.trans_datetime.dt.dayofweek
transactions_master['is_weekend'] = transactions_master.day_of_week.isin([5,6])
transactions_master['is_night'] = transactions_master.hour.isin([0,1,2,3,22,23])

In [20]:
transactions_master.head(5)

,ID,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,unix_time,merch_lat,merch_long,is_fraud,data_type,trans_datetime,hour,day_of_week,is_weekend,is_night
0,0,1/1/2019 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,1325376018,36.011293,-82.048315,0,train,2019-01-01 00:00:00,0,1,False,True
1,1,1/1/2019 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,1325376044,49.159047,-118.186462,0,train,2019-01-01 00:00:00,0,1,False,True
2,2,1/1/2019 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,1325376051,43.150704,-112.154481,0,train,2019-01-01 00:00:00,0,1,False,True
3,3,1/1/2019 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,1325376076,47.034331,-112.561071,0,train,2019-01-01 00:01:00,0,1,False,True
4,4,1/1/2019 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,1325376186,38.674999,-78.632459,0,train,2019-01-01 00:03:00,0,1,False,True


#### 3.2 Top 3 merchant concentration per card

In [25]:
q37_result = (
    transactions_master.groupby(['cc_num', 'merchant'])['amt']
    .sum()
    .reset_index()
    .assign(
        total_card_amount=lambda x: x.groupby('cc_num')['amt'].transform('sum'),
        merchant_rank=lambda x: x.groupby('cc_num')['amt'].rank(method='first', ascending=False)
    )
    .query('merchant_rank <= 3')
    .groupby('cc_num')['amt']
    .sum()
    .reset_index()
    .assign(
        top_3_concentration_pct=lambda x: round(100 * x['amt'] / 
                                               x.set_index('cc_num').index.map(
                                                   transactions_master.groupby('cc_num')['amt'].sum()
                                               ), 2)
    )
)[['cc_num', 'top_3_concentration_pct']].sort_values('top_3_concentration_pct', ascending=False)

q37_result.to_csv('Q37_top3_concentration.csv', index=False)
print("✅ Q37 saved:", q37_result.shape)
print("\n🏆 Riskiest cards:")
print(q37_result.head(10))

✅ Q37 saved: (959, 2)

🏆 Riskiest cards:
           cc_num  top_3_concentration_pct
235  3.881750e+13                    89.74
735  4.734310e+15                    88.07
359  3.756230e+14                    85.90
12   5.018950e+11                    84.96
507  3.540420e+15                    84.19
98   4.295300e+12                    81.61
488  3.529790e+15                    81.04
297  3.401870e+14                    80.19
898  4.352310e+18                    74.78
513  3.542830e+15                    73.40


#### 3.3 Top 5 cards per merchant

In [26]:
q38_result = (
    transactions_master.groupby(['merchant', 'cc_num'])['amt']
    .sum()
    .reset_index()
    .assign(
        total_merchant_amount=lambda x: x.groupby('merchant')['amt'].transform('sum'),
        cc_rank=lambda x: x.groupby('merchant')['amt'].rank(method='first', ascending=False)
    )
    .query('cc_rank <= 5')
    .groupby('merchant')['amt']
    .sum()
    .reset_index()
    .assign(
        top_5_card_concentration_pct=lambda x: round(100 * x['amt'] / 
                                                   x.set_index('merchant').index.map(
                                                       transactions_master.groupby('merchant')['amt'].sum()
                                                   ), 2)
    )
)[['merchant', 'top_5_card_concentration_pct']].sort_values('top_5_card_concentration_pct', ascending=False)

q38_result.to_csv('Q38_top5_card_concentration.csv', index=False)
print("✅ Q38 saved:", q38_result.shape)
print("\n🚨 Riskiest merchants:")
print(q38_result.head(10))

✅ Q38 saved: (693, 2)

🚨 Riskiest merchants:
                                 merchant  top_5_card_concentration_pct
71                      fraud_Boyer-Haley                         45.88
544                fraud_Satterfield-Lowe                         42.54
339                 fraud_Kozey-McDermott                         41.10
432      fraud_Monahan, Hermann and Johns                         40.13
292                 fraud_Johnston-Casper                         39.82
154     fraud_Eichmann, Hayes and Treutel                         37.58
90   fraud_Champlin, Rolfson and Connelly                         36.45
335                     fraud_Kovacek Ltd                         35.98
109                fraud_Corwin-Romaguera                         34.14
216                   fraud_Hackett Group                         33.83


#### 3.4 High Velocity Days (>10 txns/day)

In [27]:
q39_result = (
    transactions_master.groupby(['cc_num', pd.to_datetime(transactions_master.trans_date_trans_time).dt.date])['amt']
    .size()
    .reset_index(name='daily_txns')
    .query('daily_txns > 10')
    .groupby('cc_num')
    .size()
    .reset_index(name='high_velocity_days')
    .sort_values('high_velocity_days', ascending=False)
)

q39_result.to_csv('Q39_high_velocity_days.csv', index=False)
print("✅ Q39 saved:", q39_result.shape)
print("\n⚡ Most active velocity offenders:")
print(q39_result.head(10))

✅ Q39 saved: (539, 2)

⚡ Most active velocity offenders:
           cc_num  high_velocity_days
455  6.011370e+15                 249
375  4.302480e+15                 163
380  4.364010e+15                 103
256  2.720430e+15                 100
166  2.131790e+14                  99
518  4.587660e+18                  94
67   4.746000e+12                  91
475  6.538440e+15                  91
103  3.027300e+13                  91
296  3.545110e+15                  89


#### 3.5 Temporal Fraud Shift (Pre/Post 2020-01-10)

In [31]:
midpoint = pd.to_datetime('2020-01-10')

df_temp = transactions_master.copy()
df_temp['trans_date'] = pd.to_datetime(df_temp.trans_date_trans_time).dt.date
df_temp['period'] = df_temp.trans_date >= midpoint.date()

# Calculate per period per card
pre_stats = df_temp[df_temp.period == False].groupby('cc_num').agg({
    'is_fraud': ['sum', 'count']
}).droplevel(0, axis=1).reset_index()
pre_stats.columns = ['cc_num', 'pre_fraud_count', 'pre_txn_count']

post_stats = df_temp[df_temp.period == True].groupby('cc_num').agg({
    'is_fraud': ['sum', 'count']
}).droplevel(0, axis=1).reset_index()
post_stats.columns = ['cc_num', 'post_fraud_count', 'post_txn_count']

# Merge + calculate rates
q40_result = pre_stats.merge(post_stats, on='cc_num', how='outer').fillna(0)
q40_result['fraud_pre'] = round(100 * q40_result.pre_fraud_count / q40_result.pre_txn_count, 2)
q40_result['fraud_post'] = round(100 * q40_result.post_fraud_count / q40_result.post_txn_count, 2)
q40_result['fraud_delta'] = round(q40_result.fraud_post - q40_result.fraud_pre, 2)
q40_result = q40_result[['cc_num', 'fraud_pre', 'fraud_post', 'fraud_delta']].sort_values('fraud_delta', ascending=False)

q40_result.to_csv('Q40_temporal_fraud_shift.csv', index=False)
print("✅ Q40 PERFECT:", q40_result.shape)
print("\n📈 Fraud rate changes:")
print(q40_result.head(10))
print("\nSample:", q40_result.iloc[100:105])  # Mid-range cards

✅ Q40 PERFECT: (959, 4)

📈 Fraud rate changes:
           cc_num  fraud_pre  fraud_post  fraud_delta
935  4.725840e+18        0.0        5.78         5.78
751  4.874010e+15        0.0        5.61         5.61
57   6.759620e+11        0.0        5.56         5.56
150  3.001150e+13        0.0        5.50         5.50
68   6.762760e+11        0.0        5.36         5.36
79   4.026220e+12        0.0        5.34         5.34
866  6.597250e+15        0.0        5.30         5.30
25   5.702730e+11        0.0        4.89         4.89
205  3.051090e+13        0.0        4.76         4.76
482  3.527060e+15        0.0        4.64         4.64

Sample:            cc_num  fraud_pre  fraud_post  fraud_delta
109  4.481130e+12        0.0        1.71         1.71
730  4.710790e+15        0.0        1.67         1.67
554  3.563840e+15        0.0        1.67         1.67
331  3.703490e+14        0.0        1.67         1.67
709  4.607070e+15        0.0        1.66         1.66


In [7]:
df_train.head(5)

,ID,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,data_type
0,0,1/1/2019 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,-81.1781,3495,"Psychologist, counselling",3/9/1988,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,Train
1,1,1/1/2019 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,-118.2105,149,Special educational needs teacher,6/21/1978,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,Train
2,2,1/1/2019 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,-112.2620,4154,Nature conservation officer,1/19/1962,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,Train
3,3,1/1/2019 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,-112.1138,1939,Patent attorney,1/12/1967,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,Train
4,4,1/1/2019 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,-79.4629,99,Dance movement psychotherapist,3/28/1986,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,Train
